CONTENT_BASED

Ce code construit un petit système de recommandation basé sur les genres. Il commence par définir un transformer GenresBinarizer qui convertit la colonne genres_list (liste de genres par film) en colonnes numériques 0/1 utilisables par un modèle. Ensuite, il crée un pipeline scikit‑learn: d’abord un prétraitement (ColumnTransformer) qui applique GenresBinarizer aux genres et laisse passer des features agrégées (user_mean_rating, movie_mean_rating, movie_rating_count), puis un DecisionTreeClassifier qui apprend, à partir de l’historique, si un utilisateur aime (1) ou non (0) un film. Après avoir séparé les données en train/test et entraîné le pipeline, on évalue l’accuracy. Enfin, on construit une table movies avec un film par ligne et on définit recommend_for_user, qui pour un utilisateur donné supprime les films déjà vus, calcule la probabilité qu’il aime chaque film restant via le pipeline, et renvoie les films non vus avec la plus forte probabilité d’être aimés, en tenant compte aussi de leur note moyenne globale.

Importation des fichier CSV

In [81]:
import pandas as pd
import matplotlib.pyplot as plt

ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

df = ratings.merge(movies, on='movieId')

df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,16,4.0,1217897793,Casino (1995),Crime|Drama
1,1,24,1.5,1217895807,Powder (1995),Drama|Sci-Fi
2,1,32,4.0,1217896246,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,4.0,1217896556,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,4.0,1217896523,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


Nettoyage de donnees

In [82]:
df.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64

Separation de donnees

In [83]:
df["genres_list"] = df["genres"].str.split("|")

In [84]:
df = df.drop(columns=['genres'])

Mettre nouvelle colonne Like

In [85]:
df['like'] = (df["rating"] >= 3).astype(int)

In [86]:
df

,userId,movieId,rating,timestamp,title,genres_list,like
0,1,16,4.0,1217897793,Casino (1995),"[Crime, Drama]",1
1,1,24,1.5,1217895807,Powder (1995),"[Drama, Sci-Fi]",0
2,1,32,4.0,1217896246,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),"[Mystery, Sci-Fi, Thriller]",1
3,1,47,4.0,1217896556,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1
4,1,50,4.0,1217896523,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1
...,...,...,...,...,...,...,...
105334,668,142488,4.0,1451535844,Spotlight (2015),[Thriller],1
105335,668,142507,3.5,1451535889,Pawn Sacrifice (2015),[Drama],1
105336,668,143385,4.0,1446388585,Bridge of Spies (2015),"[Drama, Thriller]",1
105337,668,144976,2.5,1448656898,Bone Tomahawk (2015),"[Horror, Western]",0


Creation de colonne pour predire si le user aime ou non un film.

In [87]:
# moyenne de la note par utilisateur
user_stats = df.groupby("userId")["rating"].agg(user_mean_rating="mean")
# moyenne de la note + nombre de notes par film
movie_stats = df.groupby("movieId")["rating"].agg(
    movie_mean_rating="mean",      # moyenne des ratings pour ce film
    movie_rating_count="count"     # combien de ratings pour ce film
)

# ajoute la colonne user_mean_rating à chaque ligne selon son userId
df = df.merge(user_stats, on="userId", how="left")
# ajoute movie_mean_rating et movie_rating_count à chaque ligne selon son movieId
df = df.merge(movie_stats, on="movieId", how="left")

Si utilisateur aime ou pas un film

In [88]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# Transformer pour les genres
class GenresBinarizer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()
    def fit(self, X, y=None):
        self.mlb.fit(X.iloc[:, 0])
        return self
    def transform(self, X):
        return self.mlb.transform(X.iloc[:, 0])

# Définir X et y (on ne prend PAS timestamp, title, userId, movieId ici)
feature_cols = ["genres_list", "user_mean_rating", "movie_mean_rating", "movie_rating_count"]
X = df[feature_cols]
y = df["like"]

preprocess = ColumnTransformer(
    transformers=[
        ("genres", GenresBinarizer(), ["genres_list"]),
        ("num", "passthrough", ["user_mean_rating", "movie_mean_rating", "movie_rating_count"])
    ]
)

clf = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", clf)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
print("Accuracy:", pipe.score(X_test, y_test))

Accuracy: 0.8567970381621416


utilise le model pour generer une liste recommander pour un utilisateur, triés par probabilité d’aimer et qualité moyenne.

In [89]:
# movies uniques + stats (tu peux réutiliser movie_stats si tu veux)
movies = df.groupby("movieId", as_index=False).agg(
    title=("title", "first"),
    genres_list=("genres_list", "first"),
    movie_mean_rating=("rating", "mean"),
    movie_rating_count=("rating", "count")
)

def recommend_for_user(user_id, df_ratings, movies, pipe, top_n=10,min_votes=20):
    # ratings du user
    user_ratings = df_ratings[df_ratings["userId"] == user_id]

    # films déjà vus
    seen = set(user_ratings["movieId"])

    # films candidats
    min_votes = 3
    candidates = movies[
        (~movies["movieId"].isin(seen)) &
        (movies["movie_rating_count"] >= min_votes)
    ].copy()
    if candidates.empty:
        return candidates

    # récupérer la moyenne de rating de ce user (calculée avant)
    user_mean = df_ratings.loc[df_ratings["userId"] == user_id, "user_mean_rating"].iloc[0]

    # construire X_cand avec les mêmes features que pour l'entraînement
    X_cand = candidates.copy()
    X_cand["user_mean_rating"] = user_mean

    # on a déjà movie_mean_rating et movie_rating_count dans `candidates`
    feature_cols = ["genres_list", "user_mean_rating", "movie_mean_rating", "movie_rating_count"]
    X_cand = X_cand[feature_cols]

    proba = pipe.predict_proba(X_cand)[:, 1]
    candidates["like_proba"] = proba

    recs = candidates.sort_values(
        ["like_proba", "movie_mean_rating"],
        ascending=False
    ).head(top_n)

    return recs[["movieId", "title", "like_proba", "movie_mean_rating", "movie_rating_count"]]



Les metrics d'evaluation de recommandation de film

In [90]:
import numpy as np

def evaluate_recommender(df, movies_stats, pipe, top_n=10, min_votes=20):
    precisions = []
    hits = []

    liked_df = df[df["like"] == 1]
    user_like_counts = liked_df.groupby("userId")["movieId"].count()
    eligible_users = user_like_counts[user_like_counts >= 2].index

    for user_id in eligible_users:
        user_likes = liked_df[liked_df["userId"] == user_id]
        hidden_row = user_likes.sample(1, random_state=42)
        hidden_movie_id = hidden_row["movieId"].iloc[0]

        df_train_like = df[~((df["userId"] == user_id) & (df["movieId"] == hidden_movie_id))].copy()

        user_mean = df_train_like.groupby("userId")["rating"].mean().rename("user_mean_rating")
        df_train_like = df_train_like.drop(columns=["user_mean_rating"], errors="ignore")
        df_train_like = df_train_like.merge(user_mean, on="userId")

        movies_stats_train = df_train_like.groupby("movieId", as_index=False).agg(
            title=("title", "first"),
            genres_list=("genres_list", "first"),
            movie_mean_rating=("rating", "mean"),
            movie_rating_count=("rating", "count")
        )

        recs = recommend_for_user(
            user_id=user_id,
            df_ratings=df_train_like,
            movies=movies_stats_train,
            pipe=pipe,
            top_n=top_n,
            min_votes=min_votes
        )

        if recs is None or recs.empty:
            continue

        recommended_ids = set(recs["movieId"])
        hit = int(hidden_movie_id in recommended_ids)
        precision = hit / top_n

        hits.append(hit)
        precisions.append(precision)

    return {
        "Precision@10": np.mean(precisions) if precisions else 0,
        "Hit@10": np.mean(hits) if hits else 0
    }

In [91]:
results = evaluate_recommender(df, movies, pipe, top_n=10, min_votes=20)
results
#Dataleak, j'ai un bon score pour mon model AI, et lorsque je test les choix de films avec ceux qui devrais reellement l'etre, j'ai un accuracy score tres faible.
#ceci vien du fais que j'ai creer des colonnes comme user_rate_mean ou movie_rate_mean qui sont calculer a partir de la note de l'utilisateur pour le film, et lorsque je cache une note pour tester mon model, ces colonnes sont faussé, et du coup mon model ne peut pas faire de bonne prédiction.

{'Precision@10': np.float64(0.0004491017964071857),
 'Hit@10': np.float64(0.004491017964071856)}

Le Test de model

In [95]:
print("\nRecommandations pour l'utilisateur 2:")
print(recommend_for_user(2, df, movies, pipe))


Recommandations pour l'utilisateur 2:
      movieId                                              title  like_proba  \
1415     1809                         Fireworks (Hana-bi) (1997)         1.0   
8634    79274                  Batman: Under the Red Hood (2010)         1.0   
6456    31364         Memories of Murder (Salinui chueok) (2003)         1.0   
9339    94466                                Black Mirror (2011)         1.0   
578       670             World of Apu, The (Apur Sansar) (1959)         1.0   
1064     1306  Until the End of the World (Bis ans Ende der W...         1.0   
1807     2284                                Bandit Queen (1994)         1.0   
2082     2609              King of Masks, The (Bian Lian) (1996)         1.0   
2575     3224           Woman in the Dunes (Suna no onna) (1964)         1.0   
3961     5099                                       Heidi (1937)         1.0   

      movie_mean_rating  movie_rating_count  
1415           4.900000           

In [93]:
# toutes les lignes (notes) de l'utilisateur 2
user1_rows = df[df["userId"] == 22]

# juste les films (unique) qu'il a vus, avec le titre
user1_movies = user1_rows[["movieId", "title"]].drop_duplicates()
print(user1_movies)

      movieId                                              title
1640        3                            Grumpier Old Men (1995)
1641        5                 Father of the Bride Part II (1995)
1642       25                           Leaving Las Vegas (1995)
1643       32          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
1644       36                            Dead Man Walking (1995)
...       ...                                                ...
1836     7438                           Kill Bill: Vol. 2 (2004)
1837     8010                           Power of One, The (1992)
1838     8531                                White Chicks (2004)
1839    27728  Ghost in the Shell 2: Innocence (a.k.a. Innoce...
1840    27846                            Corporation, The (2003)

[201 rows x 2 columns]


In [94]:
import pickle

# Ici, "model" doit être ton modèle final (celui que tu veux déployer)
with open("reco_model.pkl", "wb") as f:
    pickle.dump(pipe, f)